# Metaphor recognition with decomposition models


In [1]:
from datasets import load_dataset
# we tag with spacy
def gen_texts(dataset, dataset_column_name):
    for ex in dataset:
        # each ex is a dict like {"text": "...", ...}
        txt = ex.get(dataset_column_name)
        if txt:
            # let spaCy handle casing etc.; keep original text
            yield txt

def extract_vectors(doc):
    """
    doc: spaCy Doc
    start_sent_id: int -> the id to assign to the FIRST sentence in this doc
    returns:
        rows: list of dicts, one per extracted vector
        count: how many vectors we added
        next_sent_id: the next free sentence id after this doc
    """

    S = doc.vocab.strings
    DEP_NSUBJ = S["nsubj"]
    DEP_OBJ = S["dobj"]
    DEP_IOBJ = S["dative"]
    DEP_OBL = S["obl"]
    ROOT_VERB = S["VERB"]

    rows = []
    vec_count = 0

    for sent in doc.sents:
        root = sent.root
        # check that the root POS is a verb
        if root.pos != ROOT_VERB:  # equivalent to: if root.pos_ != "VERB"
            continue

        nsubj = obj = obj2 = obl = 'None'
        filled = 0  # how many slots filled

        for child in root.children:
            d = child.dep
            if d == DEP_NSUBJ and nsubj is 'None':
                nsubj = child.lemma_
                filled += 1
            elif d == DEP_OBJ:
                if obj is 'None':
                    obj = child.lemma_
                    filled += 1
                elif obj2 is 'None':
                    obj2 = child.lemma_
                    filled += 1
            elif d == DEP_OBL and obl is 'None':
                obl = child.lemma_
                filled += 1

            if filled == 4:
                break

        root_lemma = root.lemma_

        # build the row for CSV
        vector_tuple = (root_lemma, nsubj, obj, obj2, obl)

        rows.append({
            # "root_lemma": root_lemma,
            # "nsubj": nsubj,
            # "obj": obj,
            # "obj2": obj2,
            # "obl": obl,
            "vector": vector_tuple,     ### NEW: store full tuple in one field
            "sentence_text": sent.text        ### NEW: store original sentence text
        })

        vec_count += 1

    return rows, vec_count



<>:42: SyntaxWarning: "is" with 'str' literal. Did you mean "=="?
<>:46: SyntaxWarning: "is" with 'str' literal. Did you mean "=="?
<>:49: SyntaxWarning: "is" with 'str' literal. Did you mean "=="?
<>:52: SyntaxWarning: "is" with 'str' literal. Did you mean "=="?
<>:42: SyntaxWarning: "is" with 'str' literal. Did you mean "=="?
<>:46: SyntaxWarning: "is" with 'str' literal. Did you mean "=="?
<>:49: SyntaxWarning: "is" with 'str' literal. Did you mean "=="?
<>:52: SyntaxWarning: "is" with 'str' literal. Did you mean "=="?
/tmp/ipykernel_1060453/313413645.py:42: SyntaxWarning: "is" with 'str' literal. Did you mean "=="?
  if d == DEP_NSUBJ and nsubj is 'None':
/tmp/ipykernel_1060453/313413645.py:46: SyntaxWarning: "is" with 'str' literal. Did you mean "=="?
  if obj is 'None':
/tmp/ipykernel_1060453/313413645.py:49: SyntaxWarning: "is" with 'str' literal. Did you mean "=="?
  elif obj2 is 'None':
/tmp/ipykernel_1060453/313413645.py:52: SyntaxWarning: "is" with 'str' literal. Did you mea

In [2]:
ds = load_dataset("tasksource/VUAC")
ds = ds["train"]

In [3]:
ds_metaphorical = ds.filter(lambda x:x["label"] == "metaphorical")
ds_literal = ds.filter(lambda x:x["label"] == "literal")

In [49]:
from tucker_tensor import TuckerDecomposition
from similarity import evaluate_sample
import spacy
import ast
nlp = spacy.load("en_core_web_md", disable=["ner", "textcat"])
tucker = TuckerDecomposition.load_from_disk(
    dataset="fineweb-en",
    method="siiSoftPlus",
    divergence="fr",
    dims=2000,
    rank=150,
    iterations=500
)


In [50]:
from tucker_tensor import ExtendedTucker
from utils import DATA_DIR
path = DATA_DIR/"tensors/fineweb-en/extension"
dataset="fineweb-en"
norm = ExtendedTucker.load_extensions(tucker, path/"fr_siiSoftPlus_2000d_150r_500i_norm.pt")
norm = norm.integrate_extension(None)
ext = ExtendedTucker.load_extensions(tucker, path/"fr_siiSoftPlus_2000d_150r_500i.pt")
ext = ext.integrate_extension(None)
minmax =ExtendedTucker.load_extensions(tucker, path/"fr_siiSoftPlus_2000d_150r_500i_minmax.pt")
minmax = minmax.integrate_extension(None)

{'verb': 18, 'subject': 2104, 'object': 1998}
{'verb': 18, 'subject': 2104, 'object': 1998}
{'verb': 18, 'subject': 2104, 'object': 1998}


In [60]:
import os
from similarity import load_eval_sentences_cached
vector_path = os.path.join(DATA_DIR, "vectors", "fineweb_english_vectors.csv")
sentence_sample = load_eval_sentences_cached(vector_path=vector_path,
                                             dataset="fineweb_en",
                                             seed=1,
                                             n_samples=100_000,
                                             )

In [61]:
to_check = {"tucker":tucker, "ext":ext, "norm":norm, "minmax":minmax}
for name, tensor in to_check.items():
    print(name)
    met_score = evaluate_sample(tensor, sentence_sample)
    # lit_score = evaluate_sample(tensor, sentence_sample)
    # print(met_score, lit_score, "\n~~~~~~~~~~")

tucker
Average expected role vector rank score over 100000 samples: 0.031565697839194384, Average prob score: 0.009029023349285126
Without 66604 OOV: rank 0.09451939705112702, prob 0.027036242187023163
ext
Average expected role vector rank score over 100000 samples: 0.05481608052045921, Average prob score: 0.016638342291116714
Without 42412 OOV: rank 0.09518663700850734, prob 0.0288920309394598
norm
Average expected role vector rank score over 100000 samples: 0.054816080453827964, Average prob score: 0.016638340428471565
Without 42412 OOV: rank 0.09518663689280399, prob 0.028892025351524353
minmax
Average expected role vector rank score over 100000 samples: 0.03913424892468903, Average prob score: 0.010359345935285091
Without 42412 OOV: rank 0.06795556179184732, prob 0.017988722771406174


In [51]:
met_vecs = []
met_sents = []
lit_vecs = []
lit_sents = []
for doc in nlp.pipe(gen_texts(ds_metaphorical, "context")):
    rows, count = extract_vectors(doc)
    for row in rows:
        met_vecs.append(row["vector"][:3])
        met_sents.append(row["sentence_text"])

for doc in nlp.pipe(gen_texts(ds_literal, "context")):
    rows, count = extract_vectors(doc)
    for row in rows:
        lit_vecs.append(row["vector"][:3])
        lit_sents.append(row["sentence_text"])

In [52]:
lit_test_set = [sen for sen in lit_vecs if tucker.check_vocab(sen)]
met_test_set = [sen for sen in met_vecs if tucker.check_vocab(sen)]

In [55]:
to_check = {"tucker":tucker, "ext":ext, "norm":norm, "minmax":minmax}
for name, tensor in to_check.items():
    print(name)
    met_score = evaluate_sample(tensor, met_test_set)
    lit_score = evaluate_sample(tensor, lit_test_set)
    print(met_score, lit_score, "\n~~~~~~~~~~")

tucker
Average expected role vector rank score over 2511 samples: 0.10600798116862227, Average prob score: 0.0532759428024292
Without 0 OOV: rank 0.10600798116862227, prob 0.0532759428024292
Average expected role vector rank score over 7157 samples: 0.09289175705899679, Average prob score: 0.044122107326984406
Without 0 OOV: rank 0.09289175705899679, prob 0.044122107326984406
0.10600798116862227 0.09289175705899679 
~~~~~~~~~~
ext
Average expected role vector rank score over 2511 samples: 0.07336040454585133, Average prob score: 0.026006020605564117
Without 0 OOV: rank 0.07336040454585133, prob 0.026006020605564117
Average expected role vector rank score over 7157 samples: 0.061021300268011426, Average prob score: 0.019955769181251526
Without 0 OOV: rank 0.061021300268011426, prob 0.019955769181251526
0.07336040454585133 0.061021300268011426 
~~~~~~~~~~
norm
Average expected role vector rank score over 2511 samples: 0.07336040454585133, Average prob score: 0.026006026193499565
Without 

In [42]:
met_score = evaluate_sample(tucker, met_test_set)
met_score

Average expected role vector rank score over 2144 samples: 0.10022747789337473, Average prob score: 0.056164685636758804
Without 0 OOV: rank 0.10022747789337473, prob 0.056164685636758804


np.float64(0.10022747789337473)

In [56]:
met_score = evaluate_sample(ext, met_vecs)
met_score

Average expected role vector rank score over 3477 samples: 0.05566793926614578, Average prob score: 0.01944788172841072
Without 777 OOV: rank 0.07168793512162551, prob 0.025044549256563187


np.float64(0.05566793926614578)

In [44]:
lit_score = evaluate_sample(tucker, lit_vecs)
lit_score

Average expected role vector rank score over 9320 samples: 0.056279909109160586, Average prob score: 0.028393272310495377
Without 2976 OOV: rank 0.08268107706452973, prob 0.0417126901447773


np.float64(0.056279909109160586)

In [45]:
met_score_alt = evaluate_sample(extended, met_vecs)
met_score_alt

Average expected role vector rank score over 3477 samples: 0.04764232895436176, Average prob score: 0.014224880374968052
Without 901 OOV: rank 0.06430604727263814, prob 0.019200274720788002


np.float64(0.04764232895436176)

In [46]:
lit_score_alt = evaluate_sample(extended, lit_vecs)
lit_score_alt

Average expected role vector rank score over 9320 samples: 0.04066988029786066, Average prob score: 0.010584486648440361
Without 2098 OOV: rank 0.0524845312068764, prob 0.013659292832016945


np.float64(0.04066988029786066)

In [24]:
# checking if any dimensions particularly differs between the datasets
# this would be better with a fixed verb
from collections import Counter
import torch

met_x = [v for v in met_vecs if tucker.check_vocab(v)]
met_verbs = [v[0] for v in met_x]
met_verbs_c = Counter(met_verbs)

lit_x = [v for v in lit_vecs if tucker.check_vocab(v)]
lit_verbs = [v[0] for v in lit_x]
lit_verbs_c = Counter(lit_verbs)
print("metaphorical most common verbs:", met_verbs_c.most_common(10))
print("literal most common verbs:", lit_verbs_c.most_common(10))


metaphorical most common verbs: [('go', 103), ('say', 86), ('have', 78), ('get', 70), ('come', 68), ('be', 67), ('make', 64), ('see', 57), ('take', 55), ('know', 52)]
literal most common verbs: [('say', 515), ('go', 391), ('know', 327), ('think', 311), ('want', 200), ('have', 200), ('get', 192), ('be', 155), ('come', 136), ('see', 128)]


In [25]:
from tucker_tensor import _role_index
import numpy as np

def check_met_dims(target_word, role="verb", top_k=None, print_top_dimension=False, stdev=False):
    role_index = _role_index(role)

    met_vecs_x = [vector for vector in met_x if vector[role_index] == target_word]
    lit_vecs_x = [vector for vector in lit_x if vector[role_index] == target_word]
    met_activations = [tucker.included_role_vector(v, role) for v in met_vecs_x]
    lit_activations = [tucker.included_role_vector(v, role) for v in lit_vecs_x]

    met_mat = np.array(met_activations)
    lit_mat = np.array(lit_activations)

    # we calculate the average for each dimension
    met_avg = sum(met_activations) / len(met_activations)
    lit_avg = sum(lit_activations) / len(lit_activations)
    diff = lit_avg - met_avg

    if not top_k:
        top_k = len(diff)

    print(f"{target_word}. Mean difference:", diff.mean())
    print(f"lit items: {len(lit_vecs_x)}, "
          f"met items: {len(met_vecs_x)}")
    scores, indices = torch.topk(torch.tensor(diff), top_k)
    for score, ind in zip(scores, indices):
        print(f"Dimension: {int(ind)}, score {round(float(score), 4)}")
        if stdev:
            met_dev = np.std(met_mat[:, ind])
            lit_dev = np.std(lit_mat[:, ind])
            print(f"\tdev met: {met_dev}, dev lit: {round(float(lit_dev), 4)}")
    if print_top_dimension:
        print(tucker.get_top_words_for_dimension(role, indices[0]))


In [26]:
[vector for vector in met_x if vector[0] == "see"]

[('see', 'you', 'None'),
 ('see', 'I', 'None'),
 ('see', 'None', 'None'),
 ('see', 'None', 'None'),
 ('see', 'None', 'None'),
 ('see', 'None', 'None'),
 ('see', 'he', 'None'),
 ('see', 'I', 'None'),
 ('see', 'they', 'themselves'),
 ('see', 'None', 'None'),
 ('see', 'I', 'None'),
 ('see', 'None', 'None'),
 ('see', 'None', 'None'),
 ('see', 'he', 'they'),
 ('see', 'None', 'size'),
 ('see', 'he', 'they'),
 ('see', 'you', 'None'),
 ('see', 'he', 'None'),
 ('see', 'I', 'None'),
 ('see', 'None', 'None'),
 ('see', 'None', 'None'),
 ('see', 'you', 'None'),
 ('see', 'None', 'None'),
 ('see', 'I', 'None'),
 ('see', 'he', 'None'),
 ('see', 'he', 'leg'),
 ('see', 'None', 'None'),
 ('see', 'he', 'None'),
 ('see', 'I', 'None'),
 ('see', 'I', 'None'),
 ('see', 'None', 'they'),
 ('see', 'I', 'go'),
 ('see', 'we', 'None'),
 ('see', 'I', 'None'),
 ('see', 'None', 'None'),
 ('see', 'I', 'None'),
 ('see', 'he', 'None'),
 ('see', 'I', 'None'),
 ('see', 'I', 'None'),
 ('see', 'None', 'None'),
 ('see', 'None

In [27]:
[vector for vector in lit_x if vector[0] == "see"]


[('see', 'you', 'None'),
 ('see', 'None', 'None'),
 ('see', 'you', 'None'),
 ('see', 'you', 'None'),
 ('see', 'he', 'race'),
 ('see', 'she', 'you'),
 ('see', 'he', 'they'),
 ('see', 'None', 'None'),
 ('see', 'I', 'None'),
 ('see', 'they', 'None'),
 ('see', 'he', 'None'),
 ('see', 'None', 'None'),
 ('see', 'you', 'None'),
 ('see', 'I', 'you'),
 ('see', 'you', 'None'),
 ('see', 'you', 'everything'),
 ('see', 'he', 'leg'),
 ('see', 'you', 'None'),
 ('see', 'you', 'None'),
 ('see', 'you', 'None'),
 ('see', 'who', 'None'),
 ('see', 'he', 'None'),
 ('see', 'None', 'None'),
 ('see', 'you', 'he'),
 ('see', 'he', 'None'),
 ('see', 'you', 'they'),
 ('see', 'you', 'None'),
 ('see', 'I', 'None'),
 ('see', 'None', 'None'),
 ('see', 'he', 'None'),
 ('see', 'you', 'it'),
 ('see', 'we', 'you'),
 ('see', 'they', 'None'),
 ('see', 'None', 'concert'),
 ('see', 'None', 'None'),
 ('see', 'he', 'field'),
 ('see', 'he', 'field'),
 ('see', 'None', 'None'),
 ('see', 'None', 'None'),
 ('see', 'None', 'None'),
 

In [28]:
check_met_dims("see", stdev=True, print_top_dimension=True)

see. Mean difference: 0.0001533561
lit items: 128, met items: 57
Dimension: 111, score 0.0454
	dev met: 0.03702238202095032, dev lit: 0.1434
Dimension: 121, score 0.015
	dev met: 0.008745326660573483, dev lit: 0.0393
Dimension: 55, score 0.0143
	dev met: 0.028234044089913368, dev lit: 0.0379
Dimension: 52, score 0.0095
	dev met: 0.033035941421985626, dev lit: 0.0565
Dimension: 101, score 0.0047
	dev met: 0.004112193360924721, dev lit: 0.0258
Dimension: 107, score 0.0038
	dev met: 0.0017421523807570338, dev lit: 0.0168
Dimension: 116, score 0.003
	dev met: 0.008014949969947338, dev lit: 0.0276
Dimension: 65, score 0.0024
	dev met: 0.019558534026145935, dev lit: 0.0213
Dimension: 109, score 0.0021
	dev met: 0.003733980003744364, dev lit: 0.009
Dimension: 127, score 0.0018
	dev met: 0.012178570963442326, dev lit: 0.0161
Dimension: 46, score 0.0011
	dev met: 0.004192300606518984, dev lit: 0.0068
Dimension: 92, score 0.0009
	dev met: 0.0035950897727161646, dev lit: 0.0058
Dimension: 60, sco

In [29]:
check_met_dims("role", "object", 10, print_top_dimension=True)

role. Mean difference: -0.00018694333
lit items: 3, met items: 2
Dimension: 134, score 0.0
Dimension: 51, score 0.0
Dimension: 116, score 0.0
Dimension: 70, score 0.0
Dimension: 108, score 0.0
Dimension: 21, score 0.0
Dimension: 90, score 0.0
Dimension: 54, score 0.0
Dimension: 65, score 0.0
Dimension: 99, score 0.0
[('egg', 14.852306365966797), ('mixture', 13.507144927978516), ('cup', 12.645461082458496), ('chicken', 12.433852195739746), ('water', 11.907588958740234), ('onion', 11.77444076538086), ('potato', 11.676056861877441), ('leaf', 10.938570022583008), ('bowl', 10.926586151123047), ('oil', 10.793060302734375)]
